# 03 — BIS Corpus Sentiment Extraction

**Purpose:** Run FinBERT across all BIS speech files and produce `bis_sentiment_raw.csv`.

| Path | Location |
|---|---|
| BIS speeches | `.\data\raw\Bis_Org_Speaches\` |
| Output CSV   | `.\data\processed\bis_sentiment_raw.csv` |

**Runtime:** 3–6 hours on CPU. Run overnight. Progress saves every 500 files.

---

In [1]:
import zipfile
from pathlib import Path

BIS_DIR = Path(r'.\data\raw\Bis_Org_Speaches')

zip_files = sorted(BIS_DIR.glob('speeches_*.zip'))
print(f'Found {len(zip_files)} zip files to extract\n')

for zf in zip_files:
    year = zf.stem.replace('speeches_', '')
    out_folder = BIS_DIR / year
    out_folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(out_folder)
    files_extracted = len(list(out_folder.iterdir()))
    print(f'  {zf.name}  ->  {year}/  ({files_extracted} files)')

print('\nDone. Now re-run Cell 1 to verify.')

Found 25 zip files to extract

  speeches_1997.zip  ->  1997/  (1 files)
  speeches_1998.zip  ->  1998/  (1 files)
  speeches_1999.zip  ->  1999/  (1 files)
  speeches_2000.zip  ->  2000/  (1 files)
  speeches_2001.zip  ->  2001/  (1 files)
  speeches_2002.zip  ->  2002/  (1 files)
  speeches_2003.zip  ->  2003/  (1 files)
  speeches_2004.zip  ->  2004/  (1 files)
  speeches_2005.zip  ->  2005/  (1 files)
  speeches_2006.zip  ->  2006/  (1 files)
  speeches_2007.zip  ->  2007/  (1 files)
  speeches_2008.zip  ->  2008/  (1 files)
  speeches_2009.zip  ->  2009/  (1 files)
  speeches_2010.zip  ->  2010/  (1 files)
  speeches_2011.zip  ->  2011/  (1 files)
  speeches_2012.zip  ->  2012/  (1 files)
  speeches_2013.zip  ->  2013/  (1 files)
  speeches_2014.zip  ->  2014/  (1 files)
  speeches_2015.zip  ->  2015/  (1 files)
  speeches_2016.zip  ->  2016/  (1 files)
  speeches_2017.zip  ->  2017/  (1 files)
  speeches_2018.zip  ->  2018/  (1 files)
  speeches_2019.zip  ->  2019/  (1 files)
  s

## Cell 1 — Paths and file inventory

In [3]:
from pathlib import Path
import pandas as pd

# ── PATHS ─────────────────────────────────────────────────────────
BASE_DIR    = Path(r'.')
BIS_DIR     = BASE_DIR / 'data' / 'raw' / 'Bis_Org_Speaches'
OUTPUT_DIR  = BASE_DIR / 'data' / 'processed'
OUTPUT_FILE = OUTPUT_DIR / 'bis_sentiment_raw.csv'
# ──────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()
    print('Old output file deleted.')

all_dfs = []
for csv_file in sorted(BIS_DIR.rglob('speeches_*.csv')):
    year = int(csv_file.stem.replace('speeches_', ''))
    if 1997 <= year <= 2020:
        df_yr = pd.read_csv(csv_file, encoding='utf-8')
        df_yr['year'] = year
        all_dfs.append(df_yr)

df_all = pd.concat(all_dfs, ignore_index=True)
print(f'Total speeches loaded : {len(df_all):,}')
print(f'Year range            : {df_all["year"].min()} to {df_all["year"].max()}')
print('Ready. Now run Cell 2.')

Old output file deleted.
Total speeches loaded : 16,622
Year range            : 1997 to 2020
Ready. Now run Cell 2.


In [5]:
## from pathlib import Path

BIS_DIR = Path(r'.\data\raw\Bis_Org_Speaches')

print('Inspecting contents of first 3 year folders:\n')
for year_folder in sorted(BIS_DIR.iterdir())[:3]:
    if year_folder.is_dir():
        print(f'  {year_folder.name}/')
        for f in sorted(year_folder.iterdir())[:5]:
            print(f'    {f.name}  ({f.suffix})  {f.stat().st_size/1024:.0f} KB')
        print()

Inspecting contents of first 3 year folders:

  1997/
    speeches_1997.csv  (.csv)  4449 KB

  1998/
    speeches_1998.csv  (.csv)  4015 KB

  1999/
    speeches_1999.csv  (.csv)  5651 KB



In [7]:
import pandas as pd
from pathlib import Path

BIS_DIR = Path(r'.\data\raw\Bis_Org_Speaches')

df = pd.read_csv(BIS_DIR / '1997' / 'speeches_1997.csv', encoding='utf-8')

print(f'Shape     : {df.shape}')
print(f'Columns   : {list(df.columns)}')
print(f'\nFirst row:')
print(df.iloc[0].to_string())

Shape     : (212, 6)
Columns   : ['url', 'title', 'description', 'date', 'text', 'author']

First row:
url                      https://www.bis.org/review/r970512a.pdf
title          Mr. Meyer discusses the economic outlook and t...
description    Remarks by Mr. Laurence H. Meyer, a member of ...
date                                         1997-04-24 00:00:00
text           Mr. Meyer discusses the economic outlook and t...
author                                          Laurence H Meyer


## Cell 2 — Load FinBERT and define the window-averaging scorer

In [9]:
import pandas as pd
from pathlib import Path
from collections import Counter

# ── YOUR EXACT PATHS ──────────────────────────────────────────────
BASE_DIR    = Path(r'.')
BIS_DIR     = BASE_DIR / 'data' / 'raw' / 'Bis_Org_Speaches'
OUTPUT_DIR  = BASE_DIR / 'data' / 'processed'
OUTPUT_FILE = OUTPUT_DIR / 'bis_sentiment_raw.csv'
# ─────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load all annual CSVs and combine into one dataframe
all_dfs = []
csv_files = sorted(BIS_DIR.rglob('speeches_*.csv'))

print(f'Found {len(csv_files)} annual CSV files\n')
for csv_file in csv_files:
    year = int(csv_file.stem.replace('speeches_', ''))
    if 1997 <= year <= 2020:
        df_yr = pd.read_csv(csv_file, encoding='utf-8')
        df_yr['year'] = year
        all_dfs.append(df_yr)
        print(f'  {year}  :  {len(df_yr):>4} speeches   columns: {list(df_yr.columns)}')

df_all = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal speeches loaded : {len(df_all):,}')
print(f'Year range            : {df_all["year"].min()} – {df_all["year"].max()}')
print(f'\nSample description field:')
print(df_all["description"].iloc[0])

Found 25 annual CSV files

  1997  :   212 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  1998  :   204 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  1999  :   281 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2000  :   298 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2001  :   313 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2002  :   337 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2003  :   338 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2004  :   572 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2005  :   599 speeches   columns: ['url', 'title', 'description', 'date', 'text', 'author', 'year']
  2006  :   717 speeches   columns: ['url', 'title', 'd

In [11]:
from transformers import BertTokenizer, pipeline
from tqdm import tqdm
import pandas as pd
import time

MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
clf        = pipeline('text-classification', model=MODEL_NAME, return_all_scores=True)

def score_speech(text, max_len=512, stride=50):
    text = str(text).strip()
    if not text:
        return 0.0, 0.0, 1.0
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len - 2:
        result = clf(text)[0]
        s = {r['label']: r['score'] for r in result}
        return s.get('positive',0.0), s.get('negative',0.0), s.get('neutral',0.0)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + max_len - 2, len(tokens))
        chunk_ids = ([tokenizer.cls_token_id]
                     + tokens[start:end]
                     + [tokenizer.sep_token_id])
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if end == len(tokens):
            break
        start += max_len - stride
    results = clf(chunks)
    avg = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
    for window in results:
        for item in window:
            avg[item['label']] += item['score'] / len(results)
    return avg['positive'], avg['negative'], avg['neutral']

records   = []
remaining = df_all.copy()
print(f'Speeches to score : {len(remaining):,}')
print(f'Estimated runtime : {len(remaining)/2.5/3600:.1f} hours')
print('Starting...\n')

start_time = time.time()
for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc='FinBERT', unit='speech')):
    try:
        p_pos, p_neg, p_neu = score_speech(row.text)
    except Exception:
        p_pos, p_neg, p_neu = None, None, None
    records.append({
        'url'        : row.url,
        'year'       : row.year,
        'date'       : row.date,
        'author'     : row.author,
        'description': row.description,
        'P_pos'      : round(p_pos, 6) if p_pos is not None else None,
        'P_neg'      : round(p_neg, 6) if p_neg is not None else None,
        'P_neutral'  : round(p_neu, 6) if p_neu is not None else None,
    })
    if (i + 1) % 500 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
        print(f'Saved {i+1:,} / {len(remaining):,} speeches...')

df_raw = pd.DataFrame(records)
df_raw.to_csv(OUTPUT_FILE, index=False)
elapsed = time.time() - start_time
print(f'\nDone.')
print(f'Speeches scored : {len(df_raw):,}')
print(f'Runtime         : {elapsed/3600:.2f} hours')
print(f'Saved to        : {OUTPUT_FILE}')

C:\Users\user\anaconda3\envs\finbert_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\user\anaconda3\envs\finbert_env\lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Speeches to score : 16,622
Estimated runtime : 1.8 hours
Starting...



FinBERT:   3%|█▊                                                            | 500/16622 [24:55<13:50:01,  3.09s/speech]

Saved 500 / 16,622 speeches...


FinBERT:   6%|███▋                                                          | 1000/16622 [50:08<9:21:21,  2.16s/speech]

...

Saved 16,000 / 16,622 speeches...


FinBERT:  99%|███████████████████████████████████████████████████████████▌| 16500/16622 [12:19:16<05:51,  2.88s/speech]

Saved 16,500 / 16,622 speeches...


FinBERT: 100%|████████████████████████████████████████████████████████████| 16622/16622 [12:23:45<00:00,  2.68s/speech]


Done.
Speeches scored : 16,622
Runtime         : 12.40 hours
Saved to        : .\data\processed\bis_sentiment_raw.csv


In [ ]:
if OUTPUT_FILE.exists():
    df_existing = pd.read_csv(OUTPUT_FILE)
    done_files  = set(df_existing['filename'].tolist())
    records     = df_existing.to_dict('records')
    print(f'Resuming — {len(done_files):,} speeches already scored, loading from CSV.')
else:
    done_files = set()
    records    = []
    print('No existing output found — starting fresh run.')

remaining = [f for f in all_files if f.stem not in done_files]
print(f'Speeches remaining to score : {len(remaining):,}')

import time
if len(remaining) > 0:
    est_hrs = len(remaining) / 2.5 / 3600   # ~2.5 speeches/sec on CPU
    print(f'Estimated runtime           : {est_hrs:.1f} hours at ~2.5 speeches/sec')

In [ ]:
from tqdm import tqdm

errors     = []
start_time = time.time()

for i, filepath in enumerate(tqdm(remaining, desc='FinBERT', unit='speech')):

    # Extract year from parent folder name (e.g. .../1997/r970103a.txt → 1997)
    # Falls back to None if BIS files are in a flat folder
    folder_name = filepath.parent.name
    year = int(folder_name) if folder_name.isdigit() else None

    try:
        text = filepath.read_text(encoding='utf-8', errors='ignore')
        p_pos, p_neg, p_neu = score_speech(text)
    except Exception as e:
        errors.append(str(filepath.name))
        p_pos, p_neg, p_neu = None, None, None

    records.append({
        'year'     : year,
        'filename' : filepath.stem,
        'P_pos'    : round(p_pos, 6) if p_pos is not None else None,
        'P_neg'    : round(p_neg, 6) if p_neg is not None else None,
        'P_neutral': round(p_neu, 6) if p_neu is not None else None,
    })

    # Save progress every 500 files
    if (i + 1) % 500 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)

# Final save
df_raw = pd.DataFrame(records)
df_raw.to_csv(OUTPUT_FILE, index=False)

elapsed = time.time() - start_time
print(f'\n✅  Done.')
print(f'   Speeches scored : {len(df_raw):,}')
print(f'   Errors skipped  : {len(errors)}')
print(f'   Runtime         : {elapsed/3600:.2f} hours')
print(f'   Saved to        : {OUTPUT_FILE}')

if errors:
    print(f'\n   Files that failed (first 10):')
    for e in errors[:10]:
        print(f'     {e}')

## Cell 5 — Inspect output

In [ ]:
df_raw = pd.read_csv(OUTPUT_FILE)

print(f'Shape          : {df_raw.shape}')
print(f'Year range     : {df_raw["year"].min()} – {df_raw["year"].max()}')
print(f'Missing P_neg  : {df_raw["P_neg"].isna().sum()}')

# Verify scores sum to ~1.0
df_c = df_raw.dropna(subset=['P_pos','P_neg','P_neutral']).copy()
row_sums = (df_c['P_pos'] + df_c['P_neg'] + df_c['P_neutral'] - 1.0).abs()
bad = (row_sums > 0.01).sum()
print(f'Rows where scores ≠ 1.0  : {bad}  (should be 0)')
print()
print(df_raw[['P_pos','P_neg','P_neutral']].describe().round(4))
print()
print(df_raw.head(10).to_string())
print()
print('✅  Output looks good. Proceed to Notebook 04.' if bad == 0 and df_raw["P_neg"].isna().sum() == 0
      else '⚠️  Check for missing values or score errors above.')

In [ ]:
## No Needed from transformers import BertTokenizer, pipeline
from tqdm import tqdm
import time

MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
clf        = pipeline('text-classification', model=MODEL_NAME, return_all_scores=True)

def score_speech(text, max_len=512, stride=50):
    text = str(text).strip()
    if not text:
        return 0.0, 0.0, 1.0
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len - 2:
        result = clf(text)[0]
        s = {r['label']: r['score'] for r in result}
        return s.get('positive',0.0), s.get('negative',0.0), s.get('neutral',0.0)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + max_len - 2, len(tokens))
        chunk_ids = ([tokenizer.cls_token_id]
                     + tokens[start:end]
                     + [tokenizer.sep_token_id])
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if end == len(tokens):
            break
        start += max_len - stride
    results = clf(chunks)
    avg = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
    for window in results:
        for item in window:
            avg[item['label']] += item['score'] / len(results)
    return avg['positive'], avg['negative'], avg['neutral']

# Check for existing progress
if OUTPUT_FILE.exists():
    df_existing = pd.read_csv(OUTPUT_FILE)
    done_urls   = set(df_existing['url'].tolist())
    records     = df_existing.to_dict('records')
    print(f'Resuming — {len(done_urls):,} speeches already scored.')
else:
    done_urls = set()
    records   = []
    print('Starting fresh run.')

remaining = df_all[~df_all['url'].isin(done_urls)].copy()
print(f'Speeches to score : {len(remaining):,}')
est = len(remaining) / 2.5 / 3600
print(f'Estimated runtime : {est:.1f} hours\n')

start_time = time.time()

for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc='FinBERT', unit='speech')):
    try:
        p_pos, p_neg, p_neu = score_speech(row.text)
    except Exception:
        p_pos, p_neg, p_neu = None, None, None

    records.append({
        'url'        : row.url,
        'year'       : row.year,
        'date'       : row.date,
        'author'     : row.author,
        'description': row.description,
        'P_pos'      : round(p_pos, 6) if p_pos is not None else None,
        'P_neg'      : round(p_neg, 6) if p_neg is not None else None,
        'P_neutral'  : round(p_neu, 6) if p_neu is not None else None,
    })

    if (i + 1) % 500 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)

df_raw = pd.DataFrame(records)
df_raw.to_csv(OUTPUT_FILE, index=False)

elapsed = time.time() - start_time
print(f'\n✅  Done.')
print(f'   Speeches scored : {len(df_raw):,}')
print(f'   Runtime         : {elapsed/3600:.2f} hours')
print(f'   Saved to        : {OUTPUT_FILE}')

In [ ]:
from transformers import BertTokenizer, pipeline
from tqdm import tqdm
import pandas as pd
import time

MODEL_NAME = 'ProsusAI/finbert'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
clf        = pipeline('text-classification', model=MODEL_NAME, return_all_scores=True)

def score_speech(text, max_len=512, stride=50):
    text = str(text).strip()
    if not text:
        return 0.0, 0.0, 1.0
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_len - 2:
        result = clf(text)[0]
        s = {r['label']: r['score'] for r in result}
        return s.get('positive',0.0), s.get('negative',0.0), s.get('neutral',0.0)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + max_len - 2, len(tokens))
        chunk_ids = ([tokenizer.cls_token_id]
                     + tokens[start:end]
                     + [tokenizer.sep_token_id])
        chunks.append(tokenizer.decode(chunk_ids, skip_special_tokens=True))
        if end == len(tokens):
            break
        start += max_len - stride
    results = clf(chunks)
    avg = {'positive': 0.0, 'negative': 0.0, 'neutral': 0.0}
    for window in results:
        for item in window:
            avg[item['label']] += item['score'] / len(results)
    return avg['positive'], avg['negative'], avg['neutral']

if OUTPUT_FILE.exists():
    df_existing = pd.read_csv(OUTPUT_FILE)
    done_urls   = set(df_existing['url'].tolist())
    records     = df_existing.to_dict('records')
    print(f'Resuming — {len(done_urls):,} speeches already scored.')
else:
    done_urls = set()
    records   = []
    print('Starting fresh run.')

remaining = df_all[~df_all['url'].isin(done_urls)].copy()
print(f'Speeches to score : {len(remaining):,}')
print(f'Estimated runtime : {len(remaining)/2.5/3600:.1f} hours\n')

start_time = time.time()

for i, row in enumerate(tqdm(remaining.itertuples(), total=len(remaining), desc='FinBERT', unit='speech')):
    try:
        p_pos, p_neg, p_neu = score_speech(row.text)
    except Exception:
        p_pos, p_neg, p_neu = None, None, None

    records.append({
        'url'        : row.url,
        'year'       : row.year,
        'date'       : row.date,
        'author'     : row.author,
        'description': row.description,
        'P_pos'      : round(p_pos, 6) if p_pos is not None else None,
        'P_neg'      : round(p_neg, 6) if p_neg is not None else None,
        'P_neutral'  : round(p_neu, 6) if p_neu is not None else None,
    })

    if (i + 1) % 500 == 0:
        pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)

df_raw = pd.DataFrame(records)
df_raw.to_csv(OUTPUT_FILE, index=False)
elapsed = time.time() - start_time
print(f'\n✅  Done.')
print(f'   Speeches scored : {len(df_raw):,}')
print(f'   Runtime         : {elapsed/3600:.2f} hours')
print(f'   Saved to        : {OUTPUT_FILE}')